# Session 2: LangChain Expression Language (LCEL) & Chat History

This notebook covers the advanced capabilities of LangChain for orchestration, focusing on:
1. **LangChain Expression Language (LCEL)**: Compiling prompts, models, and parsers using the `|` syntax.
2. **Custom Parsers & StrOutputParser**: Transforming model responses.
3. **Dynamic Context with RunnablePassthrough**: Injecting extra runtime keys dynamically.
4. **Runtime Bindings**: Controlling model options mid-chain.
5. **State Management**: Moving from a manual loop to `InMemoryChatMessageHistory` and `RunnableWithMessageHistory` for session-based memory management.

## Setup & Environment

Set up your environment keys.

In [ ]:
# !pip install langchain langchain-google-genai python-dotenv

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if "GOOGLE_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ImportError:
        pass

## 1. Introduction to LCEL (LangChain Expression Language)

LCEL allows you to compose chains declaratively using standard components. Let's build a simple chain linking a `ChatPromptTemplate`, a model, and a custom output sanitizer.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

sysmsg = "Imagine you're a travel assistant. Answer in no more than one line."

# Step 1: Define your prompt template
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", sysmsg),
    ("human", "{user_message}")
])

# Step 2: Define your model
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# Step 3: Define a custom sanitization function wrapped in RunnableLambda
def sanitize_manual_output(response):
    # Remove double asterisks that LLMs often use to bold words
    return response.content.strip().replace("**", "")

parser = RunnableLambda(sanitize_manual_output)

# Step 4: Compose the chain
chain = chat_prompt | llm | parser

response = chain.invoke({"user_message": "Recommend a single historic spot to visit in Rome."})
print(response)

## 2. Dynamic Context with `RunnablePassthrough`

Often, your chains require multiple input keys (e.g., retrieving dates, lookup tables, or intermediate data) that the user shouldn't need to pass manually. We use `RunnablePassthrough` to format dictionaries dynamically in the pipeline.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Define a prompt expecting both 'topic' and 'current_date'
date_prompt = ChatPromptTemplate.from_messages([
    ("system", "Today is {current_date}. Answer in exactly one sentence."),
    ("human", "What historical event happened today related to {topic}?")
])

# Build the map-chain
context_chain = (
    {
        "topic": RunnablePassthrough(), # takes the raw string passed to invoke() and maps it here
        "current_date": lambda x: "August 6"
    }
    | date_prompt
    | llm
    | StrOutputParser()
)

print(context_chain.invoke("aviation"))

## 3. Dynamic Bindings mid-chain

Instead of creating a brand new model instance with different configuration parameters, you can bind parameters (like stop tokens, temperature, or limits) directly during LCEL chain configuration.

In [ ]:
# Bind a stop sequence mid-chain
bound_chain = (
    chat_prompt 
    | llm.bind(stop=["Colosseum", "Rome"]) 
    | StrOutputParser()
)

result = bound_chain.invoke({"user_message": "Where is the Colosseum located? Answer in one sentence."})
print(f"Stopped response: {result}")

## 4. Dynamic Memory with InMemoryChatMessageHistory

Instead of writing a custom list management wrapper, we can leverage LangChain's native history components.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

chat_history = InMemoryChatMessageHistory()

chat_prompt_history = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly companion. Answer in a short phrase."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_message}")
])

chain = chat_prompt_history | llm | StrOutputParser()

print("Type 'exit' to quit.\n")
while True:
    user_msg = input("User: ")
    if user_msg.lower() == "exit":
        break
        
    response = chain.invoke({
        "user_message": user_msg,
        "chat_history": chat_history.messages
    })
    print(f"AI: {response}\n")
    
    chat_history.add_user_message(user_msg)
    chat_history.add_ai_message(response)

## 5. Session-based Memory: RunnableWithMessageHistory

To handle multiple user sessions cleanly, we can bind our chain to `RunnableWithMessageHistory`. This wraps the chain and manages session IDs automatically.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

session_store = {}

# Function to retrieve chat history for a specific session ID
def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Define the base chain
chat_prompt_session = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel planner. Remember details about the user's travel plans."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_message}"),
])

base_chain = chat_prompt_session | llm | StrOutputParser()

# Wrap base chain with RunnableWithMessageHistory
session_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="user_message",
    history_messages_key="chat_history"
)

# Interact with two different user sessions
config_user_1 = {"configurable": {"session_id": "user1"}}
config_user_2 = {"configurable": {"session_id": "user2"}}

# Session 1 Interaction
print("--- User 1 ---")
r1 = session_chain.invoke({"user_message": "Hi, my name is Alice and I am visiting Rome."}, config=config_user_1)
print(r1)

# Session 2 Interaction
print("\n--- User 2 ---")
r2 = session_chain.invoke({"user_message": "Hi, I am Bob and I want to go to Tokyo."}, config=config_user_2)
print(r2)

# Session 1 Follow up
print("\n--- User 1 Follow up ---")
r1_followup = session_chain.invoke({"user_message": "What is my name and where am I traveling?"}, config=config_user_1)
print(r1_followup)